In [11]:
import cv2
import numpy as np
import time, csv, datetime, os
import ctypes 

# ==========================================
# USER CONFIGURATION
# ==========================================
ID_CAM1 = 0
ID_CAM2 = 3
ENABLE_CAM2 = True 

# --- Grid Resolution ---
GRID_ROWS = 15
GRID_COLS = 30

# --- INITIAL RECTANGLE CONFIGURATION ---
RECT_CONFIG = {
    'CAM1': {'cx': 126, 'cy': 121, 'w': 29, 'h': 27},
    'CAM2': {'cx': 124, 'cy': 147, 'w': 43, 'h': 43}
}

# --- GEOMETRY SETTINGS ---
ANGLE_CORRECTION = {
    'CAM1': {'x': 1.0, 'y': 1.0, 'taper': 1.0}, 
    'CAM2': {'x': 1.0, 'y': 1.0, 'taper': 1.0}
}

# --- Calibration ---
OFFSETS = {
    'CAM1': 0.0,
    'CAM2': -3.4
}

EMISSIVITY = 0.95
SCALE = 3             

# ==========================================
# HELPER FUNCTIONS
# ==========================================
def is_shift_pressed():
    return (ctypes.windll.user32.GetKeyState(0x10) & 0x8000) != 0

def get_base_corners(cx, cy, w, h):
    x1 = cx - w // 2
    x2 = cx + w // 2
    y1 = cy - h // 2
    y2 = cy + h // 2
    return [(x1, y1), (x2, y1), (x2, y2), (x1, y2)]

def decode_temperatures(thermal: np.ndarray) -> np.ndarray:
    lo = thermal[..., 0].astype(np.int32)
    hi = thermal[..., 1].astype(np.int32)
    raw = lo + (hi << 8)
    temps = (raw / 64.0) - 273.15
    return temps

def generate_grid_points(corners, rows, cols, taper=1.0):
    (x0, y0), (x1, y1), (x2, y2), (x3, y3) = corners
    grid_points = []
    center_x = (x0 + x1) / 2.0
    
    for r in range(rows):
        v = r / (rows - 1) if rows > 1 else 0.5
        row_scale = 1.0 + (taper - 1.0) * v
        
        for c in range(cols):
            u = c / (cols - 1) if cols > 1 else 0.5
            
            # Linear
            xt = x0 + (x1 - x0) * u
            yt = y0 + (y1 - y0) * u
            xb = x3 + (x2 - x3) * u
            yb = y3 + (y2 - y3) * u
            x_linear = xt + (xb - xt) * v
            y_linear = yt + (yb - yt) * v
            
            # Taper
            dist_x = x_linear - center_x
            x_tapered = center_x + (dist_x * row_scale)
            
            grid_points.append((int(round(x_tapered)), int(round(y_linear))))
    return grid_points

def open_camera(index):
    cap = cv2.VideoCapture(index, cv2.CAP_MSMF)
    cap.set(cv2.CAP_PROP_FRAME_WIDTH, 256)
    cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 384)
    cap.set(cv2.CAP_PROP_FOURCC, cv2.VideoWriter_fourcc('Y','U','Y','2'))
    cap.set(cv2.CAP_PROP_CONVERT_RGB, 0)
    return cap

def process_camera_frame(cap, rect_cfg, label, is_active, offset, angle_corr):
    ret, frame = cap.read()
    if not ret: return None, None, "FAIL"

    buf = frame.ravel()
    if buf.size != 384 * 256 * 2: return None, None, "SKIP"
        
    reshaped = buf.reshape(384, 256, 2)
    visual_half, thermal_half = np.array_split(reshaped, 2)
    
    temps = decode_temperatures(thermal_half)
    temps = temps / (EMISSIVITY ** 0.25)
    temps += offset 
    temps = np.rot90(temps, 2)
    
    visual_bgr = cv2.cvtColor(visual_half, cv2.COLOR_YUV2BGR_YUYV)
    disp = cv2.resize(visual_bgr, (visual_bgr.shape[1] * SCALE, visual_bgr.shape[0] * SCALE))
    disp_color = cv2.applyColorMap(disp, cv2.COLORMAP_INFERNO)
    disp_color = cv2.rotate(disp_color, cv2.ROTATE_180)
    
    H, W = disp_color.shape[:2]
    scale_x, scale_y = W / temps.shape[1], H / temps.shape[0]
    
    # 1. Base Rect
    corners = get_base_corners(rect_cfg['cx'], rect_cfg['cy'], rect_cfg['w'], rect_cfg['h'])
    
    # 2. Draw Trapezoid
    taper_val = angle_corr.get('taper', 1.0)
    center_x = (corners[0][0] + corners[1][0]) / 2.0
    
    def apply_taper_to_x(x, v):
        scale = 1.0 + (taper_val - 1.0) * v
        dist = x - center_x
        return center_x + (dist * scale)

    poly_pts = np.array([
        [int(apply_taper_to_x(corners[0][0], 0.0) * scale_x), int(corners[0][1] * scale_y)], 
        [int(apply_taper_to_x(corners[1][0], 0.0) * scale_x), int(corners[1][1] * scale_y)], 
        [int(apply_taper_to_x(corners[2][0], 1.0) * scale_x), int(corners[2][1] * scale_y)], 
        [int(apply_taper_to_x(corners[3][0], 1.0) * scale_x), int(corners[3][1] * scale_y)]  
    ], np.int32)
    
    box_color = (0, 255, 0) if is_active else (0, 255, 255)
    thickness = 3 if is_active else 2
    cv2.polylines(disp_color, [poly_pts], True, box_color, thickness)

    # 3. Draw Grid
    grid_coords = generate_grid_points(corners, GRID_ROWS, GRID_COLS, taper=taper_val)
    
    box_temps = []
    for (gx, gy) in grid_coords:
        cv2.circle(disp_color, (int(gx*scale_x), int(gy*scale_y)), 3, (0,255,0), -1)
        if 0 <= gy < temps.shape[0] and 0 <= gx < temps.shape[1]:
            box_temps.append(temps[gy, gx])
            
    avg_temp = np.mean(box_temps) if box_temps else 0.0

    # 4. Info Label
    info_str = f"{label} | Taper: {taper_val:.1f}x | Off: {offset:+.1f}C | Avg: {avg_temp:.1f}C"
    cv2.putText(disp_color, info_str, (10, H - 20), cv2.FONT_HERSHEY_SIMPLEX, 0.6, box_color, 2)
    
    return disp_color, temps, "OK"

def main():
    recording = False
    csv_rows = []
    last_csv_time = 0
    csv_interval = 0.04 
    video_writer = None
    record_video = False
    active_cam_idx = 0 
    
    output_folder = "Data Recordings"
    os.makedirs(output_folder, exist_ok=True)

    cap1 = open_camera(ID_CAM1)
    if not cap1.isOpened(): return
    cap2 = open_camera(ID_CAM2) if ENABLE_CAM2 else None

    print("System Started. Controls: [Q]uit, [R]ecord CSV, [V]ideo, [T]Save CSV")
    
    try:
        while True:
            # 1. Process
            view1, temps1, status1 = process_camera_frame(
                cap1, RECT_CONFIG['CAM1'], "CAM 1", active_cam_idx == 0, OFFSETS['CAM1'], ANGLE_CORRECTION['CAM1'])
            
            if status1 == "SKIP": continue 
            if status1 == "FAIL": break
                
            view2, temps2 = None, None
            if ENABLE_CAM2 and cap2:
                view2, temps2, status2 = process_camera_frame(
                    cap2, RECT_CONFIG['CAM2'], "CAM 2", active_cam_idx == 1, OFFSETS['CAM2'], ANGLE_CORRECTION['CAM2'])
            
            # 2. Combine
            if view2 is not None:
                combined_view = np.hstack([view1, view2])
            else:
                combined_view = view1
            
            # 3. UI Bar
            ui_height = 100 
            ui_bar = np.zeros((ui_height, combined_view.shape[1], 3), dtype=np.uint8)
            
            rec_color = (0, 0, 255) if recording else (80, 80, 80)
            vid_color = (0, 0, 255) if record_video else (80, 80, 80)
            
            cv2.putText(ui_bar, "CSV RECORDING", (20, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.6, rec_color, 2)
            cv2.putText(ui_bar, "VIDEO RECORDING", (250, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.6, vid_color, 2)
            
            ts_display = datetime.datetime.now().strftime("%H:%M:%S.%f")[:-3]
            cv2.putText(ui_bar, ts_display, (ui_bar.shape[1] - 220, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)
            
            cmd_line1 = "Rec: [R]CSV [V]Video | Save: [T]CSV [S]Video | [Q]uit"
            cmd_line2 = "[TAB]Switch Cam | [9/0]Taper | [+/-]Offset | Arrows:Move | Shift:Resize"
            cv2.putText(ui_bar, cmd_line1, (20, 60), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (200, 200, 200), 1)
            cv2.putText(ui_bar, cmd_line2, (20, 85), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (200, 200, 200), 1)
            
            final_display = np.vstack([ui_bar, combined_view])

            if recording and int(time.time() * 2) % 2 == 0: 
                cv2.circle(final_display, (30, ui_height + 30), 10, (0, 0, 255), -1)

            cv2.imshow("Thermal Scanner", final_display)
            
            # 4. Save Logic
            if record_video:
                h, w = final_display.shape[:2]
                if video_writer is None:
                    ts = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
                    fn = os.path.join(output_folder, f"thermal_video_{ts}.avi")
                    video_writer = cv2.VideoWriter(fn, cv2.VideoWriter_fourcc(*'MJPG'), 20.0, (w, h))
                video_writer.write(final_display)

            if recording:
                now = time.time()
                if now - last_csv_time >= csv_interval:
                    row = [f"{now:.3f}".replace('.', ',')]
                    
                    # Cam 1
                    c1_phys_w = RECT_CONFIG['CAM1']['w'] * ANGLE_CORRECTION['CAM1']['x']
                    c1_phys_h = RECT_CONFIG['CAM1']['h'] * ANGLE_CORRECTION['CAM1']['y']
                    row.append(f"{c1_phys_w:.2f}".replace('.', ','))
                    row.append(f"{c1_phys_h:.2f}".replace('.', ','))
                    
                    corners1 = get_base_corners(RECT_CONFIG['CAM1']['cx'], RECT_CONFIG['CAM1']['cy'], RECT_CONFIG['CAM1']['w'], RECT_CONFIG['CAM1']['h'])
                    taper1 = ANGLE_CORRECTION['CAM1'].get('taper', 1.0)
                    pts1 = generate_grid_points(corners1, GRID_ROWS, GRID_COLS, taper=taper1)
                    for px, py in pts1:
                        val = temps1[py, px] if (0 <= py < temps1.shape[0] and 0 <= px < temps1.shape[1]) else np.nan
                        row.append(f"{val:.3f}".replace('.', ',') if np.isfinite(val) else "")

                    # Cam 2
                    if ENABLE_CAM2:
                        c2_phys_w = RECT_CONFIG['CAM2']['w'] * ANGLE_CORRECTION['CAM2']['x']
                        c2_phys_h = RECT_CONFIG['CAM2']['h'] * ANGLE_CORRECTION['CAM2']['y']
                        row.append(f"{c2_phys_w:.2f}".replace('.', ','))
                        row.append(f"{c2_phys_h:.2f}".replace('.', ','))
                        
                        if temps2 is not None:
                            corners2 = get_base_corners(RECT_CONFIG['CAM2']['cx'], RECT_CONFIG['CAM2']['cy'], RECT_CONFIG['CAM2']['w'], RECT_CONFIG['CAM2']['h'])
                            taper2 = ANGLE_CORRECTION['CAM2'].get('taper', 1.0)
                            pts2 = generate_grid_points(corners2, GRID_ROWS, GRID_COLS, taper=taper2)
                            for px, py in pts2:
                                val = temps2[py, px] if (0 <= py < temps2.shape[0] and 0 <= px < temps2.shape[1]) else np.nan
                                row.append(f"{val:.3f}".replace('.', ',') if np.isfinite(val) else "")
                    
                    csv_rows.append(row)
                    last_csv_time = now

            # --- INPUT HANDLING (FIXED) ---
            # We capture the full key code, but create a 'clean' version for checking letters.
            full_key = cv2.waitKeyEx(1)
            
            if full_key != -1:
                # Mask with 0xFF to get standard ASCII (ignores NumLock/Caps stuff)
                char_key = full_key & 0xFF
                
                # --- ARROW KEYS (Must check FULL KEY) ---
                if full_key in [2490368, 0x260000, 65362, 2621440, 0x280000, 65364, 2424832, 0x250000, 65361, 2555904, 0x270000, 65363]:
                    target = 'CAM1' if active_cam_idx == 0 else 'CAM2'
                    if full_key in [2490368, 0x260000, 65362]: # UP
                        if is_shift_pressed(): RECT_CONFIG[target]['h'] -= 2 
                        else: RECT_CONFIG[target]['cy'] -= 2 
                    elif full_key in [2621440, 0x280000, 65364]: # DOWN
                        if is_shift_pressed(): RECT_CONFIG[target]['h'] += 2 
                        else: RECT_CONFIG[target]['cy'] += 2 
                    elif full_key in [2424832, 0x250000, 65361]: # LEFT
                        if is_shift_pressed(): RECT_CONFIG[target]['w'] -= 2 
                        else: RECT_CONFIG[target]['cx'] -= 2 
                    elif full_key in [2555904, 0x270000, 65363]: # RIGHT
                        if is_shift_pressed(): RECT_CONFIG[target]['w'] += 2 
                        else: RECT_CONFIG[target]['cx'] += 2 

                # --- LETTER/NUMBER COMMANDS (Check MASKED KEY) ---
                elif char_key == ord('q') or char_key == 27: 
                    break
                elif char_key == 9: # TAB
                    active_cam_idx = 1 - active_cam_idx 
                
                # OFFSET
                elif char_key == ord('+') or char_key == 43: 
                    OFFSETS['CAM1' if active_cam_idx==0 else 'CAM2'] += 0.1
                elif char_key == ord('-') or char_key == 45: 
                    OFFSETS['CAM1' if active_cam_idx==0 else 'CAM2'] -= 0.1
                
                # TAPER
                elif char_key == ord('0') or char_key == 48: 
                    ANGLE_CORRECTION['CAM1' if active_cam_idx==0 else 'CAM2']['taper'] += 0.05
                elif char_key == ord('9') or char_key == 57: 
                    ANGLE_CORRECTION['CAM1' if active_cam_idx==0 else 'CAM2']['taper'] -= 0.05
                
                # RECORDING
                elif char_key == ord('r'): 
                    if not recording: csv_rows = []; recording = True
                elif char_key == ord('v'): 
                    if not record_video: record_video = True
                
                # SAVING
                elif char_key == ord('t'):
                    if recording:
                        recording = False
                        ts = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
                        fn = os.path.join(output_folder, f"thermal_data_{ts}.csv")
                        headers = ["timestamp", "c1_w", "c1_h"]
                        for i in range(GRID_ROWS*GRID_COLS): headers.append(f"c1_p{i}")
                        if ENABLE_CAM2:
                            headers.extend(["c2_w", "c2_h"])
                            for i in range(GRID_ROWS*GRID_COLS): headers.append(f"c2_p{i}")
                        with open(fn, "w", newline="") as f:
                            writer = csv.writer(f, delimiter=';')
                            writer.writerow(headers)
                            writer.writerows(csv_rows)
                        print(f"Saved {fn}")
                elif char_key == ord('s'): 
                    if record_video:
                        record_video = False
                        if video_writer: video_writer.release(); video_writer = None
            
    finally:
        cap1.release(); 
        if cap2: cap2.release()
        if video_writer: video_writer.release()
        cv2.destroyAllWindows()

if __name__ == "__main__":
    main()

System Started. Controls: [Q]uit, [R]ecord CSV, [V]ideo, [T]Save CSV
Saved Data Recordings\thermal_data_20260223_145622.csv
